# Notebook_05_Furusato_Analytics_Quality_and_BI

**Workshop version:** 2.7.0 optional analytics extension

Run this Notebook only after Notebook 01, or after a complete Notebook 04
provisioning run. It preserves every legacy `stg_*` and `ot_*` table used by
the Ontology and adds five schema-enabled Lakehouse areas:

- `bronze`: source-faithful data with ingestion metadata
- `silver`: conformed and deduplicated data
- `gold`: Direct Lake star schema plus an AI-friendly enriched table
- `ops`: DQ results, telemetry, and publication state
- `quarantine`: rejected increment events

The three packaged increment files intentionally contain 100 duplicate
`EventID` rows. Notebook 05 quarantines those rows and publishes 14,900 unique
events. The first run is preview-only.

Their observations all fall inside the approved synthetic window declared in
`dataset-manifest.json` (2026-08-01T00:02:47Z through 2026-08-31T23:58:20Z).
With `STRICT_SYNTHETIC_CONTRACT=True` the timestamp rule validates against that
window, so a run started before the end of the window does not quarantine the
packaged rows. Set `STRICT_SYNTHETIC_CONTRACT=False` for your own data; the rule
then falls back to rejecting timestamps in the future of the run's wall clock.

In [ ]:
# Fabric parameter cell
PARTICIPANT_ID = "001"
EXPECTED_WORKSPACE_NAME = ""
INCREMENT_PATH = "Files/increment"

APPLY_CHANGES = False
ALLOW_AUTOMATED_APPLY = False
CONFIRMED_PLAN_SHA256 = ""
EXCLUSIVE_APPLY_WINDOW_CONFIRMED = False

STRICT_SYNTHETIC_CONTRACT = True
OPERATOR_RECOVER_INTERRUPTED_RUN = False
INTERRUPTED_RUN_ID = ""

## Tested analytics extension runtime

The embedded runtime validates the schema-enabled Lakehouse, the completed
Notebook 01 source tables, the increment-data quality contract, every
run-scoped Delta candidate, and the final publication generation. No Workspace,
capacity, permission, Pipeline, Activator, or external notification change
is performed.

In [ ]:
"""Runtime for the optional Furusato Fabric analytics extension.

The extension preserves every v2.7.0 table used by the Ontology. It adds a
schema-enabled medallion view of the same data, deterministic data-quality
evidence, a quarantine surface for the intentionally duplicated increment
events, an AI-friendly enriched table, and a Direct Lake star schema.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import UTC, datetime
import hashlib
import json
import re
import uuid
from typing import Any, Mapping, Sequence


SCHEMA_NAMES = ("bronze", "silver", "gold", "ops", "quarantine")
LEGACY_SOURCE_TABLES = {
    "prefectures": "stg_prefectures",
    "municipalities": "stg_municipalities",
    "donors": "stg_donors",
    "gift_categories": "stg_categories",
    "gifts": "stg_gifts",
    "suppliers": "stg_businesses",
    "supplier_gifts": "stg_business_gifts",
    "donations": "stg_donation_orders",
}
LEGACY_GOLD_TABLES = {
    "prefecture": "ot_prefecture",
    "municipality": "ot_municipality",
    "donor": "ot_donor",
    "gift_category": "ot_gift_category",
    "gift": "ot_gift",
}
INCREMENT_SCHEMA = (
    ("EventID", "string"),
    ("DonationID", "long"),
    ("DonorID", "long"),
    ("MunicipalityID", "string"),
    ("GiftID", "long"),
    ("DonationAmountYen", "long"),
    ("DonatedAt", "timestamp"),
    ("PaymentMethod", "string"),
    ("WorkshopRunId", "string"),
    ("ParticipantAlias", "string"),
    ("SourceFile", "string"),
    ("PublishedAtUtc", "timestamp"),
)
EXPECTED_INCREMENT_CONTRACT = {
    "rawRows": 15000,
    "acceptedRows": 14900,
    "quarantinedRows": 100,
    "distinctEventIds": 14900,
}
# Approved synthetic observation window from workshop/v2.7.0/data/dataset-manifest.json
# (expectedIncrement.observationWindowUtc).  The packaged donation_events_*.csv rows are
# synthetic and deliberately dated inside this window, so validating them against the
# run's wall clock would quarantine valid rows whenever the workshop runs before the
# window closes.  STRICT_SYNTHETIC_CONTRACT therefore validates against the manifest
# window; a non-strict run keeps the wall-clock future-time guard for real data.
SYNTHETIC_OBSERVATION_WINDOW_UTC = (
    "2026-08-01T00:02:47Z",
    "2026-08-31T23:58:20Z",
)
OUTPUT_TABLES = (
    "bronze.prefectures_raw",
    "bronze.municipalities_raw",
    "bronze.donors_raw",
    "bronze.gift_categories_raw",
    "bronze.gifts_raw",
    "bronze.suppliers_raw",
    "bronze.supplier_gifts_raw",
    "bronze.donations_raw",
    "bronze.donation_events_raw",
    "silver.prefecture",
    "silver.municipality",
    "silver.donor",
    "silver.gift_category",
    "silver.gift",
    "silver.supplier",
    "silver.supplier_gift",
    "silver.donation",
    "silver.donation_event",
    "gold.prefecture",
    "gold.municipality",
    "gold.donor",
    "gold.gift_category",
    "gold.gift",
    "gold.date",
    "gold.donations",
    "gold.donation_agent",
    "ops.dq_rule_results",
    "ops.run_telemetry",
    "quarantine.donation_events_rejected",
)
CONTROL_TABLE = "ops.analytics_publish_control"
PLAN_SCHEMA_VERSION = "furusato-analytics-extension/v1"
PARTICIPANT_ID_PATTERN = re.compile(r"^(?!000$)[0-9]{3}$")


class AnalyticsExtensionError(RuntimeError):
    """Raised when the analytics extension cannot continue safely."""


@dataclass(frozen=True)
class AnalyticsExtensionConfig:
    participant_id: str
    expected_workspace_name: str = ""
    increment_path: str = "Files/increment"
    apply_changes: bool = False
    confirmed_plan_sha256: str = ""
    exclusive_apply_window_confirmed: bool = False
    strict_synthetic_contract: bool = True
    operator_recover_interrupted_run: bool = False
    interrupted_run_id: str = ""
    allow_automated_apply: bool = False


def canonical_json(value: Any) -> str:
    return json.dumps(
        value,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    )


def sha256_json(value: Any) -> str:
    return hashlib.sha256(canonical_json(value).encode("utf-8")).hexdigest()


def validate_config(config: AnalyticsExtensionConfig) -> None:
    if type(config.allow_automated_apply) is not bool:
        raise AnalyticsExtensionError("ALLOW_AUTOMATED_APPLY must be Boolean.")
    if config.allow_automated_apply and not config.expected_workspace_name.strip():
        raise AnalyticsExtensionError("Automated apply requires EXPECTED_WORKSPACE_NAME.")
    if not PARTICIPANT_ID_PATTERN.fullmatch(config.participant_id):
        raise AnalyticsExtensionError(
            "PARTICIPANT_ID must be a three-digit value from 001 through 999."
        )
    if not config.increment_path.startswith("Files/"):
        raise AnalyticsExtensionError("INCREMENT_PATH must remain under Lakehouse Files/.")
    path_parts = config.increment_path.replace("\\", "/").split("/")
    if ".." in path_parts:
        raise AnalyticsExtensionError("INCREMENT_PATH cannot contain '..'.")
    if config.apply_changes and not config.exclusive_apply_window_confirmed:
        raise AnalyticsExtensionError(
            "Set EXCLUSIVE_APPLY_WINDOW_CONFIRMED=True only after closing other "
            "writers to the analytics extension tables."
        )
    if config.operator_recover_interrupted_run and not config.interrupted_run_id:
        raise AnalyticsExtensionError(
            "INTERRUPTED_RUN_ID is required for explicit interrupted-run recovery."
        )


def build_plan(config: AnalyticsExtensionConfig) -> dict[str, Any]:
    validate_config(config)
    plan = {
        "schemaVersion": PLAN_SCHEMA_VERSION,
        "participantId": config.participant_id,
        "incrementPath": config.increment_path.replace("\\", "/").rstrip("/"),
        "schemas": list(SCHEMA_NAMES),
        "sourceTables": sorted(
            set(LEGACY_SOURCE_TABLES.values()) | set(LEGACY_GOLD_TABLES.values())
        ),
        "outputTables": list(OUTPUT_TABLES),
        "controlTable": CONTROL_TABLE,
        "strictSyntheticContract": config.strict_synthetic_contract,
        "expectedIncrementContract": EXPECTED_INCREMENT_CONTRACT,
        "calendarCoverage": "complete-calendar-years",
        "compatibility": {
            "legacyTablesPreserved": True,
            "ontologyBindingsChanged": False,
            "dataAgentFewShotCountChanged": False,
        },
    }
    if config.allow_automated_apply:
        plan["allowAutomatedApply"] = True
        plan["expectedWorkspaceName"] = config.expected_workspace_name
    return {"document": plan, "sha256": sha256_json(plan)}


def render_plan(plan: Mapping[str, Any], *, apply_mode: bool) -> list[str]:
    prefix = "APPLY_REQUESTED" if apply_mode else "PREVIEW_ONLY"
    return [
        f"{prefix}: Furusato analytics extension",
        f"PLAN_SHA256={plan['sha256']}",
        f"schemas={','.join(plan['document']['schemas'])}",
        f"sourceTables={len(plan['document']['sourceTables'])}",
        f"outputTables={len(plan['document']['outputTables'])}",
        "calendarCoverage=complete-calendar-years",
        "legacyOntologyTables=preserved",
        "fewShotCount=unchanged",
    ]


def _require(condition: bool, message: str) -> None:
    if not condition:
        raise AnalyticsExtensionError(message)


def _calendar_year_bounds(minimum: Any, maximum: Any) -> tuple[Any, Any]:
    """Classic time intelligence needs complete calendar years."""
    _require(minimum is not None and maximum is not None, "A calendar requires dated fact rows.")
    if isinstance(minimum, datetime):
        minimum = minimum.date()
    if isinstance(maximum, datetime):
        maximum = maximum.date()
    _require(minimum <= maximum, "Calendar date bounds are reversed.")
    return minimum.replace(month=1, day=1), maximum.replace(month=12, day=31)


def _require_source_tables(spark: Any) -> None:
    missing = [
        name
        for name in sorted(
            set(LEGACY_SOURCE_TABLES.values()) | set(LEGACY_GOLD_TABLES.values())
        )
        if not spark.catalog.tableExists(name)
    ]
    if missing:
        raise AnalyticsExtensionError(
            "Notebook 05 requires a completed Notebook 01 publication; "
            f"missing source tables={missing}"
        )


def _require_schema_enabled_lakehouse(spark: Any) -> None:
    rows = spark.sql("SHOW SCHEMAS").collect()
    names = {
        str(row[0]).rsplit(".", 1)[-1].strip("`").casefold()
        for row in rows
        if row and row[0] is not None
    }
    if "dbo" not in names:
        raise AnalyticsExtensionError(
            "Notebook 05 requires a schema-enabled Lakehouse. "
            "Attach the intended schema-enabled Lakehouse as the default. "
            f"Observed schema namespaces: {[str(row[0]) for row in rows if row]}"
        )


def _schema_from_contract(types_module: Any, contract: Sequence[tuple[str, str]]) -> Any:
    type_map = {
        "string": types_module.StringType(),
        "long": types_module.LongType(),
        "timestamp": types_module.TimestampType(),
    }
    return types_module.StructType(
        [
            types_module.StructField(name, type_map[data_type], False)
            for name, data_type in contract
        ]
    )


def _metadata_columns(frame: Any, functions_module: Any, *, source_file: str, batch_id: str) -> Any:
    return (
        frame.withColumn("IngestedAtUtc", functions_module.current_timestamp())
        .withColumn("SourceFile", functions_module.lit(source_file))
        .withColumn("BatchId", functions_module.lit(batch_id))
    )


def _write_control(
    spark: Any,
    functions_module: Any,
    *,
    run_id: str,
    plan_sha256: str,
    state: str,
    generation: str,
    message: str,
) -> None:
    row = [
        (
            "furusato-analytics-extension",
            run_id,
            plan_sha256,
            state,
            generation,
            datetime.now(UTC).replace(tzinfo=None),
            message,
        )
    ]
    schema = (
        "ControlKey string, RunId string, PlanSha256 string, State string, "
        "GenerationId string, UpdatedAtUtc timestamp, Message string"
    )
    frame = spark.createDataFrame(row, schema=schema)
    (
        frame.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(CONTROL_TABLE)
    )


def _read_existing_control(spark: Any) -> dict[str, Any] | None:
    if not spark.catalog.tableExists(CONTROL_TABLE):
        return None
    rows = spark.table(CONTROL_TABLE).limit(2).collect()
    if len(rows) != 1:
        raise AnalyticsExtensionError(
            f"{CONTROL_TABLE} must contain exactly one control row."
        )
    return rows[0].asDict(recursive=True)


def _validate_recovery(
    config: AnalyticsExtensionConfig,
    control: Mapping[str, Any] | None,
) -> str | None:
    if not control or str(control.get("State")) == "Ready":
        return None
    prior_run_id = str(control.get("RunId") or "")
    if not config.operator_recover_interrupted_run:
        raise AnalyticsExtensionError(
            "A prior analytics publication is not Ready. Confirm that its Spark "
            "session has stopped, then set OPERATOR_RECOVER_INTERRUPTED_RUN=True "
            f"and INTERRUPTED_RUN_ID={prior_run_id!r}."
        )
    if config.interrupted_run_id != prior_run_id:
        raise AnalyticsExtensionError(
            "INTERRUPTED_RUN_ID does not match the persisted control row."
        )
    prior_generation = str(control.get("GenerationId") or "")
    try:
        uuid.UUID(prior_generation)
    except ValueError as exc:
        raise AnalyticsExtensionError(
            "The interrupted control row has an invalid GenerationId."
        ) from exc
    return prior_generation


def _synthetic_window_bounds() -> tuple[datetime, datetime]:
    start, end = SYNTHETIC_OBSERVATION_WINDOW_UTC
    return (
        datetime.strptime(start, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=UTC),
        datetime.strptime(end, "%Y-%m-%dT%H:%M:%SZ").replace(tzinfo=UTC),
    )


def _out_of_contract_time_condition(config: AnalyticsExtensionConfig, functions_module: Any) -> Any:
    """Return the DonatedAt rejection condition for the active contract mode.

    Strict runs check the approved synthetic window from the dataset manifest so a
    workshop executed before the window closes does not quarantine packaged rows.
    Non-strict runs keep the wall-clock future-time guard.
    """
    donated_at = functions_module.col("DonatedAt")
    if not config.strict_synthetic_contract:
        return donated_at.isNull() | (donated_at > functions_module.current_timestamp())
    start, end = _synthetic_window_bounds()
    return (
        donated_at.isNull()
        | (donated_at < functions_module.lit(start.replace(tzinfo=None)).cast("timestamp"))
        | (donated_at > functions_module.lit(end.replace(tzinfo=None)).cast("timestamp"))
    )


def _time_rule_name(config: AnalyticsExtensionConfig) -> str:
    return (
        "increment.outside_synthetic_window"
        if config.strict_synthetic_contract
        else "increment.invalid_or_future_time"
    )


def _build_frames(spark: Any, config: AnalyticsExtensionConfig, *, run_id: str) -> tuple[dict[str, Any], list[tuple[Any, ...]]]:
    from pyspark.sql import Window
    from pyspark.sql import functions as F
    from pyspark.sql import types as T

    batch_id = f"analytics-{config.participant_id}-{run_id}"
    bronze = {
        f"bronze.{target}_raw": _metadata_columns(
            spark.table(source),
            F,
            source_file=f"legacy-table:{source}",
            batch_id=batch_id,
        )
        for target, source in LEGACY_SOURCE_TABLES.items()
    }

    increment_path = config.increment_path.rstrip("/") + "/*.csv"
    event_schema = _schema_from_contract(T, INCREMENT_SCHEMA)
    event_raw = (
        spark.read.option("header", True)
        .schema(event_schema)
        .csv(increment_path)
        .withColumn("InputFilePath", F.input_file_name())
        .withColumn("IngestedAtUtc", F.current_timestamp())
        .withColumn("BatchId", F.lit(batch_id))
    )
    bronze["bronze.donation_events_raw"] = event_raw

    silver = {
        "silver.prefecture": spark.table("stg_prefectures").dropDuplicates(
            ["PrefectureID"]
        ),
        "silver.municipality": spark.table("stg_municipalities").dropDuplicates(
            ["MunicipalityID"]
        ),
        "silver.donor": spark.table("stg_donors").dropDuplicates(["DonorID"]),
        "silver.gift_category": spark.table("stg_categories").dropDuplicates(
            ["CategoryID"]
        ),
        "silver.gift": spark.table("stg_gifts").dropDuplicates(["GiftID"]),
        "silver.supplier": spark.table("stg_businesses").dropDuplicates(
            ["BusinessID"]
        ),
        "silver.supplier_gift": spark.table("stg_business_gifts").dropDuplicates(
            ["BusinessID", "GiftID"]
        ),
        "silver.donation": spark.table("stg_donation_orders").dropDuplicates(
            ["DonationID"]
        ),
    }

    invalid_time_condition = _out_of_contract_time_condition(config, F)
    invalid_time_reason = (
        "OUTSIDE_SYNTHETIC_WINDOW"
        if config.strict_synthetic_contract
        else "INVALID_OR_FUTURE_DONATED_AT"
    )

    event_ranked = event_raw.withColumn(
        "__DuplicateOrdinal",
        F.row_number().over(
            Window.partitionBy("EventID").orderBy(
                F.col("PublishedAtUtc").asc(),
                F.col("SourceFile").asc(),
                F.col("InputFilePath").asc(),
            )
        ),
    )
    donor_keys = silver["silver.donor"].select(
        F.col("DonorID").alias("__KnownDonorID")
    )
    municipality_keys = silver["silver.municipality"].select(
        F.col("MunicipalityID").alias("__KnownMunicipalityID")
    )
    gift_keys = silver["silver.gift"].select(
        F.col("GiftID").alias("__KnownGiftID")
    )
    checked = (
        event_ranked.join(
            donor_keys,
            F.col("DonorID") == F.col("__KnownDonorID"),
            "left",
        )
        .join(
            municipality_keys,
            F.col("MunicipalityID") == F.col("__KnownMunicipalityID"),
            "left",
        )
        .join(
            gift_keys,
            F.col("GiftID") == F.col("__KnownGiftID"),
            "left",
        )
        .withColumn(
            "RejectReason",
            F.concat_ws(
                "|",
                F.when(F.col("__DuplicateOrdinal") > 1, F.lit("DUPLICATE_EVENT_ID")),
                F.when(
                    F.col("DonationAmountYen").isNull()
                    | (F.col("DonationAmountYen") <= 0),
                    F.lit("INVALID_DONATION_AMOUNT"),
                ),
                F.when(
                    invalid_time_condition,
                    F.lit(invalid_time_reason),
                ),
                F.when(F.col("__KnownDonorID").isNull(), F.lit("UNKNOWN_DONOR")),
                F.when(
                    F.col("__KnownMunicipalityID").isNull(),
                    F.lit("UNKNOWN_MUNICIPALITY"),
                ),
                F.when(F.col("__KnownGiftID").isNull(), F.lit("UNKNOWN_GIFT")),
            ),
        )
    )
    quarantine = (
        checked.where(F.length("RejectReason") > 0)
        .withColumn("RejectedAtUtc", F.current_timestamp())
        .drop("__KnownDonorID", "__KnownMunicipalityID", "__KnownGiftID")
    )
    accepted = (
        checked.where(F.length("RejectReason") == 0)
        .drop(
            "__DuplicateOrdinal",
            "__KnownDonorID",
            "__KnownMunicipalityID",
            "__KnownGiftID",
            "RejectReason",
        )
    )
    silver["silver.donation_event"] = accepted

    raw_rows = event_raw.count()
    distinct_event_ids = event_raw.select("EventID").distinct().count()
    quarantined_rows = quarantine.count()
    accepted_rows = accepted.count()
    duplicate_rows = checked.where(F.col("__DuplicateOrdinal") > 1).count()
    invalid_amount_rows = checked.where(
        F.col("DonationAmountYen").isNull() | (F.col("DonationAmountYen") <= 0)
    ).count()
    invalid_time_rows = checked.where(invalid_time_condition).count()
    unknown_donor_rows = checked.where(F.col("__KnownDonorID").isNull()).count()
    unknown_municipality_rows = checked.where(
        F.col("__KnownMunicipalityID").isNull()
    ).count()
    unknown_gift_rows = checked.where(F.col("__KnownGiftID").isNull()).count()

    if config.strict_synthetic_contract:
        observed = {
            "rawRows": raw_rows,
            "acceptedRows": accepted_rows,
            "quarantinedRows": quarantined_rows,
            "distinctEventIds": distinct_event_ids,
        }
        _require(
            observed == EXPECTED_INCREMENT_CONTRACT,
            "Increment DQ contract mismatch: "
            f"expected={EXPECTED_INCREMENT_CONTRACT}, observed={observed}",
        )
        _require(
            duplicate_rows == EXPECTED_INCREMENT_CONTRACT["quarantinedRows"],
            "The synthetic duplicate-event exercise no longer has 100 duplicates.",
        )
        _require(
            all(
                value == 0
                for value in (
                    invalid_amount_rows,
                    invalid_time_rows,
                    unknown_donor_rows,
                    unknown_municipality_rows,
                    unknown_gift_rows,
                )
            ),
            "The canonical increment files contain an unexpected non-duplicate error. "
            "Every packaged DonatedAt must fall inside the approved synthetic window "
            f"{SYNTHETIC_OBSERVATION_WINDOW_UTC[0]} .. {SYNTHETIC_OBSERVATION_WINDOW_UTC[1]}.",
        )

    static_fact = silver["silver.donation"].select(
        F.col("DonationID").alias("DonationId"),
        F.col("DonationAmountYen"),
        F.col("DonatedAt").alias("DonatedAtUtc"),
        F.to_date(F.from_utc_timestamp("DonatedAt", "Asia/Tokyo")).alias(
            "DonationDate"
        ),
        F.col("PaymentMethod"),
        F.lit("StaticSeed").alias("DataSource"),
        (F.col("DonationAmountYen") > F.lit(57000)).alias("IsHighValue"),
        F.col("DonorID").alias("DonorId"),
        F.col("MunicipalityID").alias("MunicipalityId"),
        F.col("GiftID").alias("GiftId"),
    )
    increment_fact = accepted.select(
        F.col("DonationID").alias("DonationId"),
        F.col("DonationAmountYen"),
        F.col("DonatedAt").alias("DonatedAtUtc"),
        F.to_date(F.from_utc_timestamp("DonatedAt", "Asia/Tokyo")).alias(
            "DonationDate"
        ),
        F.col("PaymentMethod"),
        F.lit("RealtimeIncrement").alias("DataSource"),
        (F.col("DonationAmountYen") > F.lit(57000)).alias("IsHighValue"),
        F.col("DonorID").alias("DonorId"),
        F.col("MunicipalityID").alias("MunicipalityId"),
        F.col("GiftID").alias("GiftId"),
    )
    donation_fact = static_fact.unionByName(increment_fact)

    gold_prefecture = spark.table(LEGACY_GOLD_TABLES["prefecture"]).select(
            "PrefectureId",
            "PrefectureName",
            "PrefectureNameEn",
        )
    gold_gift_category = spark.table(
        LEGACY_GOLD_TABLES["gift_category"]
    ).select(
            "CategoryId",
            "CategoryName",
            "CategoryNameEn",
            "CategorySearchTerms",
        )
    gold = {
        "gold.prefecture": gold_prefecture,
        "gold.municipality": (
            spark.table(LEGACY_GOLD_TABLES["municipality"])
            .alias("m")
            .join(
                gold_prefecture.alias("p"),
                F.col("m.PrefectureId") == F.col("p.PrefectureId"),
            )
            .select(
                F.col("m.MunicipalityId"),
                F.col("m.MunicipalityName"),
                F.col("m.MunicipalityDisplayName"),
                F.col("m.PrefectureId"),
                F.col("p.PrefectureName"),
                F.col("p.PrefectureNameEn"),
            )
        ),
        "gold.donor": (
            spark.table(LEGACY_GOLD_TABLES["donor"])
            .alias("d")
            .join(
                gold_prefecture.alias("p"),
                F.col("d.PrefectureId") == F.col("p.PrefectureId"),
            )
            .select(
                F.col("d.DonorId"),
                F.col("d.DonorName"),
                F.col("d.DonorDisplayName"),
                F.col("d.DonorAge"),
                F.col("d.DonorOccupation"),
                F.col("d.DonorOccupationEn"),
                F.col("d.PrefectureId"),
                F.col("p.PrefectureName").alias("ResidencePrefectureName"),
                F.col("p.PrefectureNameEn").alias("ResidencePrefectureNameEn"),
            )
        ),
        "gold.gift_category": gold_gift_category,
        "gold.gift": (
            spark.table(LEGACY_GOLD_TABLES["gift"])
            .alias("g")
            .join(
                gold_gift_category.alias("c"),
                F.col("g.CategoryId") == F.col("c.CategoryId"),
            )
            .select(
                F.col("g.GiftId"),
                F.col("g.GiftName"),
                F.col("g.GiftDisplayName"),
                F.col("g.GiftSearchTerms"),
                F.col("g.CategoryId"),
                F.col("c.CategoryName"),
                F.col("c.CategoryNameEn"),
                F.col("g.MunicipalityId"),
            )
        ),
        "gold.donations": donation_fact,
    }
    date_bounds = donation_fact.agg(
        F.min("DonationDate").alias("MinDate"),
        F.max("DonationDate").alias("MaxDate"),
    ).first()
    calendar_start, calendar_end = _calendar_year_bounds(date_bounds["MinDate"], date_bounds["MaxDate"])
    gold["gold.date"] = (
        spark.range(1).select(
            F.explode(F.sequence(F.lit(calendar_start), F.lit(calendar_end))).alias("Date")
        )
        .withColumn("DateKey", F.date_format("Date", "yyyyMMdd").cast("long"))
        .withColumn("Year", F.year("Date").cast("long"))
        .withColumn("Quarter", F.concat(F.lit("Q"), F.quarter("Date")))
        .withColumn("MonthNumber", F.month("Date").cast("long"))
        .withColumn("MonthNameJa", F.concat(F.month("Date"), F.lit("月")))
        .withColumn("YearMonth", F.date_format("Date", "yyyy-MM"))
        .select(
            "DateKey",
            "Date",
            "Year",
            "Quarter",
            "MonthNumber",
            "MonthNameJa",
            "YearMonth",
        )
    )
    gold["gold.donation_agent"] = (
        donation_fact.alias("f")
        .join(
            gold["gold.donor"].alias("d"),
            F.col("f.DonorId") == F.col("d.DonorId"),
        )
        .join(
            gold["gold.municipality"].alias("m"),
            F.col("f.MunicipalityId") == F.col("m.MunicipalityId"),
        )
        .join(
            gold["gold.gift"].alias("g"),
            F.col("f.GiftId") == F.col("g.GiftId"),
        )
        .select(
            F.col("f.DonationId"),
            F.col("f.DonationAmountYen"),
            F.col("f.DonatedAtUtc"),
            F.col("f.DonationDate"),
            F.col("f.PaymentMethod"),
            F.col("f.DataSource"),
            F.col("f.IsHighValue"),
            F.col("d.DonorId"),
            F.col("d.DonorName"),
            F.col("d.DonorAge"),
            F.col("d.DonorOccupation"),
            F.col("m.MunicipalityId"),
            F.col("m.MunicipalityName"),
            F.col("m.PrefectureId"),
            F.col("m.PrefectureName"),
            F.col("g.GiftId"),
            F.col("g.GiftName"),
            F.col("g.CategoryId"),
            F.col("g.CategoryName"),
        )
    )

    dq_rows = [
        ("increment.raw_rows", "Info", EXPECTED_INCREMENT_CONTRACT["rawRows"], raw_rows),
        (
            "increment.duplicate_event_id",
            "Warning",
            EXPECTED_INCREMENT_CONTRACT["quarantinedRows"],
            duplicate_rows,
        ),
        ("increment.invalid_amount", "Error", 0, invalid_amount_rows),
        (_time_rule_name(config), "Error", 0, invalid_time_rows),
        ("increment.unknown_donor", "Error", 0, unknown_donor_rows),
        ("increment.unknown_municipality", "Error", 0, unknown_municipality_rows),
        ("increment.unknown_gift", "Error", 0, unknown_gift_rows),
    ]
    evaluated_at = datetime.now(UTC).replace(tzinfo=None)
    dq_frame = spark.createDataFrame(
        [
            (
                run_id,
                rule_name,
                severity,
                expected_failed,
                actual_failed,
                "PASS" if expected_failed == actual_failed else "FAIL",
                evaluated_at,
            )
            for rule_name, severity, expected_failed, actual_failed in dq_rows
        ],
        schema=(
            "RunId string, RuleName string, Severity string, "
            "ExpectedValue long, ActualValue long, Status string, "
            "EvaluatedAtUtc timestamp"
        ),
    )
    failed_error_rules = dq_frame.where(
        (F.col("Severity") == "Error") & (F.col("Status") == "FAIL")
    ).count()
    _require(failed_error_rules == 0, "One or more Error-severity DQ rules failed.")

    frames = {
        **bronze,
        **silver,
        **gold,
        "ops.dq_rule_results": dq_frame,
        "quarantine.donation_events_rejected": quarantine,
    }
    telemetry_rows = [
        (
            run_id,
            name,
            "Prepared",
            int(frame.count()),
            datetime.now(UTC).replace(tzinfo=None),
        )
        for name, frame in sorted(frames.items())
    ]
    frames["ops.run_telemetry"] = spark.createDataFrame(
        telemetry_rows,
        schema=(
            "RunId string, StepName string, Status string, RowCount long, "
            "RecordedAtUtc timestamp"
        ),
    )
    return frames, dq_rows


def _temp_name(final_name: str, generation: str) -> str:
    schema_name, table_name = final_name.split(".", 1)
    token = generation.replace("-", "")[:12]
    return f"{schema_name}.__tmp_{token}_{table_name}"


def _temp_table_map(generation: str) -> dict[str, str]:
    return {
        final_name: _temp_name(final_name, generation)
        for final_name in OUTPUT_TABLES
    }


def _with_generation(frame: Any, functions_module: Any, generation: str) -> Any:
    return frame.withColumn("_AnalyticsGenerationId", functions_module.lit(generation))


def _require_published_generation(
    actual_generations: Sequence[str],
    *,
    row_count: int,
    generation: str,
    table_name: str,
) -> None:
    expected_generations = [] if row_count == 0 else [generation]
    _require(
        list(actual_generations) == expected_generations,
        f"Published generation mismatch for {table_name}: "
        f"expected={expected_generations}, actual={list(actual_generations)}.",
    )


def _drop_temp_tables(
    spark: Any,
    temp_tables: Mapping[str, str],
    *,
    operation: str,
) -> None:
    failures: list[str] = []
    for temp_name in temp_tables.values():
        try:
            spark.sql(f"DROP TABLE IF EXISTS {temp_name}")
        except Exception as exc:
            failures.append(f"{temp_name}: {type(exc).__name__}: {exc}")
    if failures:
        raise AnalyticsExtensionError(
            f"Failed to {operation} one or more run-scoped tables: {failures}"
        )


def execute_analytics_extension(
    spark: Any,
    config: AnalyticsExtensionConfig,
    *,
    workspace_name: str = "",
    is_interactive: bool = True,
) -> dict[str, Any]:
    """Preview or apply the optional schema-enabled analytics extension."""
    from pyspark.sql import functions as F

    validate_config(config)
    if (
        config.expected_workspace_name
        and workspace_name != config.expected_workspace_name
    ):
        raise AnalyticsExtensionError(
            f"Workspace mismatch: expected {config.expected_workspace_name!r}, "
            f"got {workspace_name!r}."
        )
    _require_schema_enabled_lakehouse(spark)
    _require_source_tables(spark)

    plan = build_plan(config)
    for line in render_plan(plan, apply_mode=config.apply_changes):
        print(line)
    if not config.apply_changes:
        return {
            "mode": "preview",
            "planSha256": plan["sha256"],
            "outputTables": list(OUTPUT_TABLES),
        }
    if not is_interactive and not config.allow_automated_apply:
        raise AnalyticsExtensionError(
            "Analytics extension writes require an interactive run or explicit "
            "ALLOW_AUTOMATED_APPLY=True in the confirmed preview."
        )
    if config.confirmed_plan_sha256.strip().casefold() != plan["sha256"].casefold():
        raise AnalyticsExtensionError(
            "CONFIRMED_PLAN_SHA256 does not match this run. "
            f"Expected {plan['sha256']}."
        )

    for schema_name in SCHEMA_NAMES:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema_name}`")
    existing_control = _read_existing_control(spark)
    interrupted_generation = _validate_recovery(config, existing_control)
    if interrupted_generation is not None:
        _drop_temp_tables(
            spark,
            _temp_table_map(interrupted_generation),
            operation="clean the explicitly acknowledged interrupted generation",
        )

    run_id = str(uuid.uuid4())
    generation = str(uuid.uuid4())
    frames, _ = _build_frames(spark, config, run_id=run_id)
    _require(
        set(frames) == set(OUTPUT_TABLES),
        "The generated table set does not match the published output contract.",
    )
    temp_tables = _temp_table_map(generation)
    expected_row_counts: dict[str, int] = {}
    _write_control(
        spark,
        F,
        run_id=run_id,
        plan_sha256=plan["sha256"],
        state="Preparing",
        generation=generation,
        message="Run-scoped Delta candidate materialization started.",
    )
    try:
        for final_name, temp_name in temp_tables.items():
            (
                _with_generation(frames[final_name], F, generation)
                .write.format("delta")
                .mode("errorifexists")
                .saveAsTable(temp_name)
            )

        for final_name, temp_name in temp_tables.items():
            expected_rows = frames[final_name].count()
            actual_rows = spark.table(temp_name).count()
            expected_row_counts[final_name] = expected_rows
            _require(
                expected_rows == actual_rows,
                f"Materialized row-count mismatch for {final_name}: "
                f"expected={expected_rows}, actual={actual_rows}",
            )
    except Exception as exc:
        try:
            _drop_temp_tables(
                spark,
                temp_tables,
                operation="roll back failed candidate preparation",
            )
        except AnalyticsExtensionError as cleanup_error:
            raise AnalyticsExtensionError(
                f"Candidate preparation failed and cleanup was incomplete: "
                f"{cleanup_error}"
            ) from exc
        raise

    _write_control(
        spark,
        F,
        run_id=run_id,
        plan_sha256=plan["sha256"],
        state="Publishing",
        generation=generation,
        message="All run-scoped Delta candidates passed validation.",
    )
    for final_name, temp_name in temp_tables.items():
        spark.sql(
            f"CREATE OR REPLACE TABLE {final_name} USING DELTA "
            f"AS SELECT * FROM {temp_name}"
        )
        published = spark.table(final_name)
        published_row_count = published.count()
        _require(
            published_row_count == expected_row_counts[final_name],
            f"Published row-count mismatch for {final_name}: "
            f"expected={expected_row_counts[final_name]}, "
            f"actual={published_row_count}",
        )
        published_generations = [
            row[0]
            for row in published
            .select("_AnalyticsGenerationId")
            .distinct()
            .collect()
        ]
        _require_published_generation(
            published_generations,
            row_count=published_row_count,
            generation=generation,
            table_name=final_name,
        )

    _write_control(
        spark,
        F,
        run_id=run_id,
        plan_sha256=plan["sha256"],
        state="Ready",
        generation=generation,
        message=(
            f"Published {len(OUTPUT_TABLES)} tables; "
            f"acceptedEvents={EXPECTED_INCREMENT_CONTRACT['acceptedRows']}; "
            f"quarantinedEvents={EXPECTED_INCREMENT_CONTRACT['quarantinedRows']}."
        ),
    )
    _drop_temp_tables(
        spark,
        temp_tables,
        operation="clean the successfully published generation",
    )

    return {
        "mode": "applied",
        "planSha256": plan["sha256"],
        "runId": run_id,
        "generationId": generation,
        "outputTables": list(OUTPUT_TABLES),
        "controlTable": CONTROL_TABLE,
    }

## Preview or apply

Leave `APPLY_CHANGES=False` for the first run and copy the printed
`PLAN_SHA256`. Then set `APPLY_CHANGES=True`,
`CONFIRMED_PLAN_SHA256` to that exact value, and
`EXCLUSIVE_APPLY_WINDOW_CONFIRMED=True`.
For a Jobs API run, explicitly set `ALLOW_AUTOMATED_APPLY=True` and
`EXPECTED_WORKSPACE_NAME` before previewing. This changes the plan hash;
the interactive default and all other apply gates remain unchanged.

If an earlier Spark session stopped during publication, confirm that session
is no longer running. Use the exact persisted `RunId` printed by the error with
`OPERATOR_RECOVER_INTERRUPTED_RUN=True`; never guess the value.

In [ ]:
context = notebookutils.runtime.context
workspace_name = context["currentWorkspaceName"]

CONFIG = AnalyticsExtensionConfig(
    participant_id=PARTICIPANT_ID,
    expected_workspace_name=EXPECTED_WORKSPACE_NAME,
    increment_path=INCREMENT_PATH,
    apply_changes=APPLY_CHANGES,
    allow_automated_apply=ALLOW_AUTOMATED_APPLY,
    confirmed_plan_sha256=CONFIRMED_PLAN_SHA256,
    exclusive_apply_window_confirmed=EXCLUSIVE_APPLY_WINDOW_CONFIRMED,
    strict_synthetic_contract=STRICT_SYNTHETIC_CONTRACT,
    operator_recover_interrupted_run=OPERATOR_RECOVER_INTERRUPTED_RUN,
    interrupted_run_id=INTERRUPTED_RUN_ID,
)

ANALYTICS_RESULT = execute_analytics_extension(
    spark,
    CONFIG,
    workspace_name=workspace_name,
    is_interactive=bool(context.get("isForInteractive", False)),
)

print(f"ANALYTICS_MODE={ANALYTICS_RESULT['mode']}")
print(f"PLAN_SHA256={ANALYTICS_RESULT['planSha256']}")
if ANALYTICS_RESULT["mode"] == "applied":
    print(f"GENERATION_ID={ANALYTICS_RESULT['generationId']}")
    print("ANALYTICS_READY: Direct Lake and AI consumption tables are published.")